# Inspect trained BEV-Gemma QA model

Loads checkpoints from `vla.bev_qa.train` and prints question / ground-truth / prediction on validation samples.

Edit `OUTPUT_DIR`, `QA_DIR`, and `IMG_ROOT` below, then run cells in order.

In [1]:
!hostname
!nvidia-smi

gpu26


Tue May 19 14:25:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:61:00.0 Off |                  Off |
| 30%   25C    P8             29W /  300W |       1MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+


In [2]:
import json
import os
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "vla").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find repo root (no 'vla/' package) starting from {start.resolve()}"
    )


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT={REPO_ROOT}")

from vla.bev_qa.dataloader import BEVQADataset, bev_qa_collate_fn
from vla.bev_qa.model import BEVGemmaQA
from vla.bev_qa.train import _resolve_dtype, _split_indices

OUTPUT_DIR = Path("/zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma")
QA_DIR = "/zfsauton/scratch/mineuih/waymax_rs/qa_dataset/"
IMG_ROOT = "/zfsauton/scratch/eshau/imgs_past/"
FILE_INDICES = None
VALIDATION_FRACTION = 0.04
SEED = 0

NUM_EXAMPLES = 100
MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 8
BATCH_SIZE = 1

# None = latest checkpoint; or set e.g. 27055 for step_00027055.pt
CHECKPOINT_STEP = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = _resolve_dtype("bf16") if device.type == "cuda" else torch.float32
print(f"device={device}, dtype={dtype}")


REPO_ROOT=/zfsauton2/home/sbellad/waymax_rs-main


device=cuda, dtype=torch.bfloat16


In [3]:


def load_bev_training_config(output_dir: Path, checkpoint: dict) -> dict:
    config_path = output_dir / "config.json"
    if config_path.exists():
        with open(config_path, encoding="utf-8") as f:
            return json.load(f)
    if "config" in checkpoint:
        return checkpoint["config"]
    raise FileNotFoundError(f"No config.json in {output_dir} and checkpoint has no config.")


def find_checkpoint(output_dir: Path, step: int | None = None) -> Path:
    ckpt_dir = output_dir / "checkpoints"
    if step is not None:
        path = ckpt_dir / f"step_{int(step):08d}.pt"
        if not path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        return path

    checkpoints = sorted(ckpt_dir.glob("step_*.pt"))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")
    return checkpoints[-1]


checkpoint_path = find_checkpoint(OUTPUT_DIR, CHECKPOINT_STEP)
print(f"Loading checkpoint: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
train_config = load_bev_training_config(OUTPUT_DIR, checkpoint)

GEMMA_NAME = train_config.get("gemma_name", "google/gemma-4-E2B-it")
FREEZE_GEMMA = train_config.get("freeze_gemma", True)
FREEZE_VISION = train_config.get("freeze_vision", True)
MAX_PROMPT_LENGTH = train_config.get("max_prompt_length", MAX_PROMPT_LENGTH)
MAX_ANSWER_LENGTH = train_config.get("max_answer_length", MAX_ANSWER_LENGTH)
VALIDATION_FRACTION = train_config.get("validation_fraction", VALIDATION_FRACTION)
SEED = train_config.get("seed", SEED)
QA_DIR = train_config.get("qa_dir", QA_DIR)
IMG_ROOT = train_config.get("img_root", IMG_ROOT)
FILE_INDICES = train_config.get("file_indices", FILE_INDICES)

model = BEVGemmaQA(
    gemma_name=GEMMA_NAME,
    freeze_gemma=FREEZE_GEMMA,
    freeze_vision=FREEZE_VISION,
)

if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

state_key = "trainable_model_state_dict"
if state_key not in checkpoint:
    raise KeyError(
        f"Checkpoint missing {state_key!r}. Keys: {list(checkpoint.keys())}"
    )

missing, unexpected = model.load_state_dict(
    checkpoint[state_key],
    strict=False,
)
if missing:
    print(f"Warning: {len(missing)} missing keys (expected for frozen Gemma weights)")
if unexpected:
    raise RuntimeError(f"Unexpected keys in checkpoint: {unexpected}")

model = model.to(device=device, dtype=dtype)
model.eval()

print("Model loaded successfully.")
print(f"Checkpoint step: {checkpoint.get('step', 'unknown')}")
print(f"gemma_name={GEMMA_NAME}")
print(f"trainable tensors loaded: {len(checkpoint[state_key])}")


Loading checkpoint: /zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma/checkpoints/step_00027055.pt


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[BEVGemmaQA] hidden_size=1536
[BEVGemmaQA] vision_hidden_size=768
[BEVGemmaQA] vision_soft_tokens_per_image=280


Model loaded successfully.
Checkpoint step: 27055
gemma_name=google/gemma-4-E2B-it
trainable tensors loaded: 12


In [4]:
dataset = BEVQADataset(
    qa_dir=QA_DIR,
    img_root=IMG_ROOT,
    file_indices=FILE_INDICES,
    one_qa_per_scenario=True,
    seed=SEED,
)

_, val_indices = _split_indices(len(dataset), VALIDATION_FRACTION, SEED)
inspection_indices = val_indices if val_indices else list(range(len(dataset)))
inspection_ds = Subset(dataset, inspection_indices)

loader = DataLoader(
    inspection_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=bev_qa_collate_fn,
)

print(f"dataset samples={len(dataset)}")
print(f"inspection samples={len(inspection_ds)} (validation_fraction={VALIDATION_FRACTION})")
print(f"missing_images={dataset.missing_images}, skipped_no_qas={dataset.skipped_no_qas}")

shown = 0
correct = 0

def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().strip().split())


with torch.inference_mode():
    for batch in loader:
        remaining = NUM_EXAMPLES - shown
        if remaining <= 0:
            break

        images = batch["images"][:remaining]
        questions = batch["questions"][:remaining]
        answers = batch["answers"][:remaining]
        qa_keys = batch["qa_keys"][:remaining]

        amp_enabled = device.type == "cuda"
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
            predictions = model.generate_answers(
                images=images,
                questions=questions,
                max_prompt_length=MAX_PROMPT_LENGTH,
                max_new_tokens=MAX_ANSWER_LENGTH,
            )

        for question, answer, prediction, qa_key in zip(
            questions, answers, predictions, qa_keys
        ):
            match = _normalize_text(prediction) == _normalize_text(answer)
            correct += int(match)

            print(f"[key={qa_key}]")
            print(f"Q:   {question}")
            print(f"GT:  {answer}")
            print(f"Pred:{prediction}")
            print("-" * 88)
            shown += 1

        if shown >= NUM_EXAMPLES:
            break

accuracy = correct / shown if shown > 0 else 0.0
print(f"\nExact-match accuracy on {shown} examples: {accuracy:.3f} ({correct}/{shown})")


Loading BEV-QA data:   0%|                                                                                                                                                       | 0/130 [00:00<?, ?file/s]


Loading BEV-QA data:   1%|▊                                                                                                          | 1/130 [00:00<00:15,  8.32file/s, missing=0, samples=410, skipped=45]


Loading BEV-QA data:   2%|██▍                                                                                                      | 3/130 [00:00<00:13,  9.73file/s, missing=0, samples=1312, skipped=136]


Loading BEV-QA data:   4%|████                                                                                                     | 5/130 [00:00<00:12, 10.36file/s, missing=0, samples=2201, skipped=221]


Loading BEV-QA data:   5%|█████▋                                                                                                   | 7/130 [00:00<00:11, 10.70file/s, missing=0, samples=3084, skipped=319]

...

[key=num_vehicle_front_same_lane]
Q:   How many vehicles are in front of the ego vehicle in the same lane?
GT:  0
Pred:0.000 0.
----------------------------------------------------------------------------------------


[key=num_vehicle_behind_same_lane]
Q:   How many vehicles are behind the ego vehicle in the same lane?
GT:  0
Pred:1.1310 -1
----------------------------------------------------------------------------------------


[key=traffic_light_state]
Q:   What is the traffic light state class? Answer with the integer class.
GT:  -1
Pred:-1.00 -1.
----------------------------------------------------------------------------------------


[key=num_vehicle_right]
Q:   How many vehicles are on the right side of the ego vehicle?
GT:  0
Pred:0.0000 0
----------------------------------------------------------------------------------------


[key=num_vehicle_right]
Q:   How many vehicles are on the right side of the ego vehicle?
GT:  0
Pred:0.0000 0
----------------------------------------------------------------------------------------

Exact-match accuracy on 100 examples: 0.000 (0/100)


In [5]:
ckpt_dir = OUTPUT_DIR / "checkpoints"
checkpoints = sorted(ckpt_dir.glob("step_*.pt"))
print(f"Found {len(checkpoints)} checkpoints in {ckpt_dir}")
for path in checkpoints:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"  {path.name}  step={ckpt.get('step', '?')}")

Found 28 checkpoints in /zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma/checkpoints


  step_00001000.pt  step=1000


  step_00002000.pt  step=2000


  step_00003000.pt  step=3000


  step_00004000.pt  step=4000


...

  step_00023000.pt  step=23000


  step_00024000.pt  step=24000


  step_00025000.pt  step=25000


  step_00026000.pt  step=26000


  step_00027000.pt  step=27000
  step_00027055.pt  step=27055


## Cleaner answers (new cells only)

The cells above use plain `generate_answers` with a short `max_new_tokens` budget. The model often continues past the correct token (`yes yes yes`, `0.0000 0`, `vehicle - 10.01`).

Below we try three complementary fixes **without changing training code**:

1. **Post-hoc parsing** — extract the first valid yes/no, integer, float, `x y`, or object type from the raw string.
2. **Extra `eos_token_id`s** — stop as soon as Gemma emits `yes`, `no`, a digit, `vehicle`, etc. ([HF `generate` docs](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationConfig.eos_token_id)).
3. **Forced choice (logits mask)** — for yes/no (and small integers), only allow tokens in a small allowed set at each generation step.

In [6]:
import re
from typing import Any

from transformers import LogitsProcessor, LogitsProcessorList

# One schema per qa_key (matches vla/bev_qa/dataloader.py templates).
QA_SCHEMAS: dict[str, dict[str, Any]] = {
    "has_left_lane": {"kind": "yes_no"},
    "has_right_lane": {"kind": "yes_no"},
    "num_vehicle_left": {"kind": "integer"},
    "num_vehicle_right": {"kind": "integer"},
    "num_vehicle_front_same_lane": {"kind": "integer"},
    "num_vehicle_behind_same_lane": {"kind": "integer"},
    "num_pedestrian_front": {"kind": "integer"},
    "traffic_light_state": {"kind": "integer"},
    "target_speed": {"kind": "float"},
    "target_heading": {"kind": "float"},
    "target_type": {"kind": "target_type"},
    "target_position": {"kind": "xy"},
}

TARGET_TYPES = ("vehicle", "pedestrian", "cyclist", "other")


def _token_ids(tokenizer, texts: list[str]) -> list[int]:
    ids: set[int] = set()
    for text in texts:
        for tid in tokenizer.encode(text, add_special_tokens=False):
            ids.add(int(tid))
    return sorted(ids)


def _base_eos_ids(tokenizer) -> list[int]:
    out: list[int] = []
    if tokenizer.eos_token_id is not None:
        out.append(int(tokenizer.eos_token_id))
    return out


def stop_token_ids_for_key(tokenizer, qa_key: str) -> list[int]:
    """Extra eos_token_id values: generation stops right after one of these tokens."""
    schema = QA_SCHEMAS.get(qa_key, {"kind": "raw"})
    stop = set(_base_eos_ids(tokenizer))

    kind = schema["kind"]
    if kind == "yes_no":
        stop.update(_token_ids(tokenizer, ["yes", "no", "\n"]))
    elif kind == "integer":
        stop.update(_token_ids(tokenizer, [str(i) for i in range(16)] + ["\n"]))
    elif kind == "float":
        stop.update(_token_ids(tokenizer, [".", "\n", "-"]))
        stop.update(_token_ids(tokenizer, [str(i) for i in range(10)]))
    elif kind == "target_type":
        stop.update(_token_ids(tokenizer, list(TARGET_TYPES) + ["\n", " "]))
    elif kind == "xy":
        stop.update(_token_ids(tokenizer, [".", "\n", "-", " "]))
        stop.update(_token_ids(tokenizer, [str(i) for i in range(10)]))

    return sorted(stop)


def allowed_token_ids_for_key(tokenizer, qa_key: str) -> list[int] | None:
    """Token mask for forced-choice decoding. None = no mask."""
    schema = QA_SCHEMAS.get(qa_key, {"kind": "raw"})
    kind = schema["kind"]

    if kind == "yes_no":
        return _token_ids(tokenizer, ["yes", "no"])
    if kind == "integer":
        return _token_ids(tokenizer, [str(i) for i in range(16)])
    return None


class AllowedTokensAtStep(LogitsProcessor):
    """Only allow tokens in `allowed_ids` (for short structured answers)."""

    def __init__(self, allowed_ids: list[int]) -> None:
        self.allowed_ids = sorted(set(int(x) for x in allowed_ids))

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        mask = torch.full_like(scores, float("-inf"))
        mask[:, self.allowed_ids] = scores[:, self.allowed_ids]
        return mask


# --- Post-hoc parsers (run on any raw string) ---

def parse_answer(qa_key: str, raw: str) -> str:
    schema = QA_SCHEMAS.get(qa_key, {"kind": "raw"})
    kind = schema["kind"]
    text = raw.strip()

    if kind == "yes_no":
        m = re.search(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
        return m.group(1).lower() if m else text.split()[0].lower()

    if kind == "integer":
        m = re.search(r"\b(\d+)\b", text)
        return m.group(1) if m else text.split()[0]

    if kind == "float":
        m = re.search(r"[-+]?\d*\.?\d+", text)
        if not m:
            return text.split()[0]
        return _format_float(m.group(0))

    if kind == "target_type":
        low = text.lower()
        for name in TARGET_TYPES:
            if name in low:
                return name
        return text.split()[0].lower()

    if kind == "xy":
        nums = re.findall(r"[-+]?\d*\.?\d+", text)
        if len(nums) >= 2:
            return f"{_format_float(nums[0])} {_format_float(nums[1])}"
        if len(nums) == 1:
            return _format_float(nums[0])
        return text.split()[0]

    return text


def _format_float(value: str | float, ndigits: int = 2) -> str:
    try:
        return f"{float(value):.{ndigits}f}"
    except Exception:
        return str(value)


# Quick sanity checks on strings from your run log
_examples = [
    ("has_left_lane", "yes yes yes yes"),
    ("num_vehicle_front_same_lane", "0.0000 0"),
    ("target_type", "vehicle - 10.01"),
    ("target_position", "6.33 10."),
]
print("Parser smoke test:")
for key, raw in _examples:
    print(f"  {key:30s} raw={raw!r:25s} -> parsed={parse_answer(key, raw)!r}")

Parser smoke test:
  has_left_lane                  raw='yes yes yes yes'         -> parsed='yes'
  num_vehicle_front_same_lane    raw='0.0000 0'                -> parsed='0'
  target_type                    raw='vehicle - 10.01'         -> parsed='vehicle'
  target_position                raw='6.33 10.'                -> parsed='6.33 10.00'


In [7]:
@torch.no_grad()
def generate_bev_answers(
    model: BEVGemmaQA,
    images: list[Any],
    questions: list[str],
    qa_keys: list[str],
    *,
    max_prompt_length: int = 128,
    max_new_tokens: int = 8,
    mode: str = "raw",
) -> list[str]:
    """Same path as model.generate_answers, with optional stop tokens / forced choice.

    mode:
      - raw: default Gemma eos only
      - stop: extra eos_token_id per qa_key (stop right after yes / 0 / vehicle / ...)
      - forced: logits mask to allowed tokens only (yes/no, small integers)
      - stop+forced: both
    """
    if mode not in {"raw", "stop", "forced", "stop+forced"}:
        raise ValueError(f"Unknown mode={mode!r}")

    was_training = model.training
    model.eval()

    device = next(model.parameters()).device
    tokenizer = model.tokenizer

    prompt_ids, prompt_mask, _, _ = model._tokenize_text(
        questions=questions,
        answers=None,
        device=device,
        max_prompt_length=max_prompt_length,
        max_answer_length=max_new_tokens,
    )

    image_tokens = model._encode_bev_images(images, device=device)
    scene_tokens = model._project_bev_tokens(image_tokens)

    batch_size = prompt_ids.shape[0]
    num_scene_tokens = scene_tokens.shape[1]

    dummy_id = tokenizer.pad_token_id
    if dummy_id is None:
        dummy_id = tokenizer.eos_token_id

    scene_dummy_ids = torch.full(
        (batch_size, num_scene_tokens),
        fill_value=dummy_id,
        dtype=prompt_ids.dtype,
        device=device,
    )
    scene_mask = torch.ones(
        (batch_size, num_scene_tokens),
        dtype=prompt_mask.dtype,
        device=device,
    )

    input_ids = torch.cat([prompt_ids, scene_dummy_ids], dim=1)
    attention_mask = torch.cat([prompt_mask, scene_mask], dim=1)

    model._scene_tokens_for_hook = scene_tokens
    model._scene_start_for_hook = prompt_ids.shape[1]
    model._scene_len_for_hook = num_scene_tokens

    predictions: list[str] = []
    try:
        for i in range(batch_size):
            row_input = input_ids[i : i + 1]
            row_mask = attention_mask[i : i + 1]
            qa_key = qa_keys[i]

            gen_kwargs: dict[str, Any] = {
                "input_ids": row_input,
                "attention_mask": row_mask,
                "max_new_tokens": max_new_tokens,
                "do_sample": False,
                "pad_token_id": tokenizer.pad_token_id,
                "use_cache": False,
            }

            eos_ids = _base_eos_ids(tokenizer)
            if mode in {"stop", "stop+forced"}:
                eos_ids = sorted(set(eos_ids) | set(stop_token_ids_for_key(tokenizer, qa_key)))
            gen_kwargs["eos_token_id"] = eos_ids[0] if len(eos_ids) == 1 else eos_ids

            if mode in {"forced", "stop+forced"}:
                allowed = allowed_token_ids_for_key(tokenizer, qa_key)
                if allowed:
                    gen_kwargs["logits_processor"] = LogitsProcessorList(
                        [AllowedTokensAtStep(allowed)]
                    )

            generated_ids = model.llm.generate(**gen_kwargs)
            input_len = row_input.shape[1]
            new_ids = generated_ids[:, input_len:]
            predictions.append(
                tokenizer.decode(new_ids[0], skip_special_tokens=True).strip()
            )
    finally:
        model._scene_tokens_for_hook = None
        model._scene_start_for_hook = None
        model._scene_len_for_hook = None

    if was_training:
        model.train()

    return predictions

In [8]:
# Re-run a subset with raw vs stop vs forced vs parse-only.
# Uses `loader` from the cell above if you already built it; otherwise rebuilds (slow).

NUM_COMPARE = 30
COMPARE_MODES = ("raw", "stop", "forced", "stop+forced")

if "loader" not in globals():
    dataset = BEVQADataset(
        qa_dir=QA_DIR,
        img_root=IMG_ROOT,
        file_indices=FILE_INDICES,
        one_qa_per_scenario=True,
        seed=SEED,
    )
    _, val_indices = _split_indices(len(dataset), VALIDATION_FRACTION, SEED)
    inspection_indices = val_indices if val_indices else list(range(len(dataset)))
    loader = DataLoader(
        Subset(dataset, inspection_indices),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        collate_fn=bev_qa_collate_fn,
    )


def _acc(preds: list[str], gts: list[str]) -> float:
    ok = sum(
        _normalize_text(p) == _normalize_text(g)
        for p, g in zip(preds, gts)
    )
    return ok / len(gts) if gts else 0.0


rows: list[dict[str, Any]] = []
shown = 0

with torch.inference_mode():
    for batch in loader:
        if shown >= NUM_COMPARE:
            break

        images = batch["images"]
        questions = batch["questions"]
        answers = batch["answers"]
        qa_keys = batch["qa_keys"]

        amp_enabled = device.type == "cuda"
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
            by_mode = {
                mode: generate_bev_answers(
                    model,
                    images=images,
                    questions=questions,
                    qa_keys=qa_keys,
                    max_prompt_length=MAX_PROMPT_LENGTH,
                    max_new_tokens=MAX_ANSWER_LENGTH,
                    mode=mode,
                )
                for mode in COMPARE_MODES
            }

        for q, gt, key, raw, stop, forced, both in zip(
            questions,
            answers,
            qa_keys,
            by_mode["raw"],
            by_mode["stop"],
            by_mode["forced"],
            by_mode["stop+forced"],
        ):
            parsed_from_raw = parse_answer(key, raw)
            rows.append(
                {
                    "qa_key": key,
                    "gt": gt,
                    "raw": raw,
                    "parsed": parsed_from_raw,
                    "stop": stop,
                    "forced": forced,
                    "stop+forced": both,
                }
            )

            print(f"[key={key}]")
            print(f"Q:          {q}")
            print(f"GT:         {gt}")
            print(f"Raw:        {raw}")
            print(f"Parsed:     {parsed_from_raw}")
            print(f"Stop-EOS:   {stop}")
            print(f"Forced:     {forced}")
            print(f"Stop+Force: {both}")
            print("-" * 88)
            shown += 1
            if shown >= NUM_COMPARE:
                break

# Accuracy summary
all_gts = [r["gt"] for r in rows]
print("\nExact-match accuracy:")
print(f"  raw output:           {_acc([r['raw'] for r in rows], all_gts):.3f}")
print(f"  parse(raw):           {_acc([r['parsed'] for r in rows], all_gts):.3f}")
print(f"  stop eos:             {_acc([r['stop'] for r in rows], all_gts):.3f}")
print(f"  forced choice:        {_acc([r['forced'] for r in rows], all_gts):.3f}")
print(f"  stop+forced:          {_acc([r['stop+forced'] for r in rows], all_gts):.3f}")
print(f"  parse(stop):          {_acc([parse_answer(r['qa_key'], r['stop']) for r in rows], all_gts):.3f}")

[key=has_left_lane]
Q:          Is there a lane on the left of the ego vehicle? Answer yes or no.
GT:         no
Raw:        yes
-1
-1
Parsed:     yes
Stop-EOS:   yes
Forced:     yesyesyesyesyesyesyesyes
Stop+Force: yes
----------------------------------------------------------------------------------------


[key=num_vehicle_front_same_lane]
Q:          How many vehicles are in front of the ego vehicle in the same lane?
GT:         0
Raw:        0.0000 0
Parsed:     0
Stop-EOS:   0
Forced:     00000000
Stop+Force: 0
----------------------------------------------------------------------------------------


[key=has_left_lane]
Q:          Is there a lane on the left of the ego vehicle? Answer yes or no.
GT:         no
Raw:        no
-10. 1
Parsed:     no
Stop-EOS:   no
Forced:     nononononononono
Stop+Force: no
----------------------------------------------------------------------------------------


[key=traffic_light_state]
Q:          What is the traffic light state class? Answer with the integer class.
GT:         6
Raw:        6.33 10.
Parsed:     6
Stop-EOS:   6
Forced:     60346666
Stop+Force: 6
----------------------------------------------------------------------------------------


[key=traffic_light_state]
Q:          What is the traffic light state class? Answer with the integer class.
GT:         -1
Raw:        -10.0 -10
Parsed:     10
Stop-EOS:   -1
Forced:     40466666
Stop+Force: 4
----------------------------------------------------------------------------------------


...

[key=target_speed]
Q:          What is the target object's speed?
GT:         0.00
Raw:        0.00 0.0
Parsed:     0.00
Stop-EOS:   0
Forced:     0.00 0.0
Stop+Force: 0
----------------------------------------------------------------------------------------


[key=num_pedestrian_front]
Q:          How many pedestrians are in front of the ego vehicle?
GT:         0
Raw:        0.000 - 0
Parsed:     0
Stop-EOS:   0
Forced:     00000000
Stop+Force: 0
----------------------------------------------------------------------------------------


[key=target_speed]
Q:          What is the target object's speed?
GT:         0.55
Raw:        0.00 0.0
Parsed:     0.00
Stop-EOS:   0
Forced:     0.00 0.0
Stop+Force: 0
----------------------------------------------------------------------------------------


[key=has_left_lane]
Q:          Is there a lane on the left of the ego vehicle? Answer yes or no.
GT:         yes
Raw:        yes
-1.1.
Parsed:     yes
Stop-EOS:   yes
Forced:     yesyesyesyesyesyesyesyes
Stop+Force: yes
----------------------------------------------------------------------------------------


[key=num_vehicle_front_same_lane]
Q:          How many vehicles are in front of the ego vehicle in the same lane?
GT:         0
Raw:        0.000 0.
Parsed:     0
Stop-EOS:   0
Forced:     00000000
Stop+Force: 0
----------------------------------------------------------------------------------------

Exact-match accuracy:
  raw output:           0.000
  parse(raw):           0.533
  stop eos:             0.533
  forced choice:        0.000
  stop+forced:          0.500
  parse(stop):          0.533


In [9]:
# Inspect which token ids we stop on for a few keys (useful when tuning eos lists).
for key in ["has_left_lane", "num_vehicle_left", "target_type", "target_position"]:
    ids = stop_token_ids_for_key(model.tokenizer, key)
    pieces = []
    for tid in ids[:20]:
        pieces.append(repr(model.tokenizer.decode([tid])))
    print(f"{key}: {len(ids)} stop ids, sample decode -> {', '.join(pieces)}")

has_left_lane: 4 stop ids, sample decode -> '<eos>', '\n', 'no', 'yes'
num_vehicle_left: 12 stop ids, sample decode -> '<eos>', '\n', '1', '0', '2', '3', '5', '4', '9', '6', '8', '7'
target_type: 9 stop ids, sample decode -> '<eos>', '\n', 'ist', 'other', 'ped', 'cycl', 'vehicle', 'estrian', ' '
target_position: 15 stop ids, sample decode -> '<eos>', '\n', ' ', '.', '1', '0', '-', '2', '3', '5', '4', '9', '6', '8', '7'
